Barren plateau gradient variance experiment setup.


In [ ]:
from qiskit import *
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.primitives import StatevectorEstimator 
from functools import partial

import torch
from torch.autograd import Function
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
import quimb as qu

import random
import time
import secrets
from EGATE import *
from NNVQE_HEA_half_uni import *

In [ ]:
import os
def generate_random_hamiltonian_graph(N=5, num_data=20):
    A = np.linspace(-3.0,3.0,num_data)
    if N==2:
        edges = [(0,1)]
    else:
        edges = [(i, i+1) for i in range(N-1)]
        edges.append((N-1, 0))
    edge_features = {}
    rand_vec = torch.rand(3)  # sample in [0,1], scale to [0,2], then shift to [-1,1]
    
    # Set the first and second values to 1
    rand_vec[0] = 1.0
    rand_vec[1] = 1.0
    rand_vec[2] = 1.0
    for e in edges:
        # edge_features[e] = torch.randint(-1, 2, (3,), dtype=torch.float32)  
        edge_features[e] = rand_vec
    return edges, edge_features

In [ ]:
sum_of_energy = []
def train(n, d, edge_full_data, latent_data, latent_size, NN_shape, maxiter=1, lr=0.1, stddev=1.0, index=0):
    model = NN_MERA_Model( n, d, stddev, NN_shape, latent_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    decay_steps = 700
    decay_rate = 0.7
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: decay_rate ** (step // decay_steps)
    )
    
    for i in range(1, maxiter + 1):
        optimizer.zero_grad()
        total_energy = torch.tensor(0.0, dtype=torch.float32)
        for j in range(1):
            edge_data = [t.tolist() for t in edge_full_data[j].values()]
            edges = list(edge_full_data[j].keys())
            graph_latent = latent_data[j]
            energy = model(edges, edge_data, graph_latent)
            total_energy += energy
        total_energy.backward()
        grad_param = QuantumCircuitFunction.grad_params_buffer

        optimizer.step()
        scheduler.step()
        sum_of_energy.append(total_energy.item())
        if i % 10 == 0:
            print(f"Epoch {i}, Total Energy: {total_energy.item()}")
    # index = secrets.randbelow(10)
    return grad_param



In [ ]:
# Define training parameters
n_list = [3,4,5,6,7,8,9]          # Number of qubits
d = [1,2,3,4,5,6,7]  
NN_shape = 20
stddev = 1.0
maxiter = 1

iter = 500
results_dict = {}  # Dictionary storing repeated scalar results for each n
var_dict = {}
var_mean_list = []

# Start training
for n in n_list:
  result_for_this_n = []
  # Build H-graph latent vector
  latent_data = []
  edge_full_data = []
  if __name__ == "__main__":
      torch.manual_seed(1)
      np.random.seed(1)
      random.seed(1)
      node_feats = one_hot_encoding_nodes(n)
      edges, edge_feats_dict = generate_random_hamiltonian_graph(n,1)
      edge_full_data.append(edge_feats_dict)
      model = EGATEAutoEncoder(
          node_in_dim= n,
          edge_in_dim= 3,
          node_hidden_dim= n,  
          edge_hidden_dim= 3,
          num_layers=3,
          decoder_hidden_dim=16,
          lambda_param=0.5,  # Lambda
          num_edges=len(edges)
      )
      num_epochs = 500
      H_rec, E_rec, mse_list, graph_latent = train_single_graph(
        model, edges, edge_feats_dict, node_feats, num_epochs=num_epochs
      )
      # # print(mse_list)
      latent_data.append(graph_latent)
      print(latent_data)
      # for k, e in enumerate(edges):
      #   print(f"  Edge {e}: org={edge_feats_dict[e]}, rec={E_rec[k]}")
  latent_size = n + 3
  _, param_num = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None}, n, d[n-3], param_num=True)
  print("qubit: {}, # of parameter: {}".format(n, param_num))
  old_py_state = random.getstate()
  random.seed(None)

  # index = secrets.randbelow(param_num) 
  index = random.randrange(0, param_num) 
  random.setstate(old_py_state)
  for i in range(iter):
    a = train(n, d[n-3], edge_full_data, latent_data, latent_size, NN_shape, maxiter,0.1, stddev, index)
    result_for_this_n.append(a)

  var_list_n=[]
  for j in range(param_num):
      temp_gar = []
      for i in range(iter):
          temp_gar.append(result_for_this_n[i][j])
      var_list_n.append(np.var(temp_gar, ddof=1))
  print(np.mean(var_list_n))

  var_mean_list.append(np.mean(var_list_n))
  var_dict[n] = var_list_n
  results_dict[n] = result_for_this_n

In [ ]:
print("GAE_var_avg = ", var_mean_list)

In [ ]:
print("GAE_var_dict = ", var_dict)

In [ ]:
print("GAE_var_mean_list = ", var_mean_list)

In [ ]:
plt.plot(n_list,var_mean_list)
plt.legend()
plt.show()


In [ ]:

stds = [np.std(var_dict[n]) for n in n_list]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(n_list, var_mean_list, linewidth=2, label='Mean')
plt.fill_between(n_list, np.array(var_mean_list) - np.array(stds), np.array(var_mean_list) + np.array(stds), alpha=0.2)

xtick_labels = [f"{val}({val-2})" for val in n_list]
plt.xticks(n_list, xtick_labels)
plt.ylabel("var[grad.]")
plt.xlabel("# of qubit (ansatz depth)")
plt.title('GAE')
plt.legend()
plt.tight_layout()
plt.show()